<div style="max-width:100%;box-sizing:border-box;border-top:4px solid #0f766e;padding:24px 0">
<div style="color:#0f766e;font-weight:700">EXTENSION · DATA WAREHOUSING WITH APACHE DORIS</div>
<h1>Module 2–3 扩展实验：批次、分区与 Profile</h1>
<p>目标 Doris 4.1.3 · 独立 ext_* 实验表</p>
</div>

[扩展入口](README.md) · [主线学习目录](../README.md)

完成主线 Lab 2、3 后运行。使用课程单容器沙箱，生成固定 100,000 行教学数据，不混入 WWI 或业务指标。
仅重建 `ext_small_batches`、`ext_large_batch`、`ext_partitioned` 三张表；不要在两个内核中并发重跑同一实验库。
建议 25–35 分钟，不计入原视频时长。先看 SQL，再运行；失败后保留输出，重新从初始化开始只重置这三张表。

验收：三张表数据完全一致；指定一天 10,000 行、金额 100,000.00；提交计划、批次/版本与 Profile 观察。
不要求小批次必定积压，不凭一次耗时宣布优化倍数。

In [ ]:
from pathlib import Path
import sys
COURSE_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))
from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import expect, fixture, normalized
from dw_course.ui import show_sql
lab = connect_sandbox()
lab.sql("SELECT VERSION() AS version")
lab.sql("SHOW BACKENDS")

## 1. 固定输入与表结构

每 10,000 行为一天，金额固定 10.00。前两张表结构相同，仅写入批次不同。
第三张保持键列、分桶一致，只增加日分区；这样不同时改变排序键与分桶数。

In [ ]:
from time import perf_counter
fields = "order_date DATE NOT NULL, order_id BIGINT NOT NULL, amount DECIMAL(12,2) NOT NULL"
partitions = ",".join(f"PARTITION p{day:02} VALUES [('2026-01-{day:02}'), ('2026-01-{day+1:02}'))" for day in range(1,11))
for table in ("ext_small_batches", "ext_large_batch", "ext_partitioned"):
    lab.execute("DROP TABLE IF EXISTS " + table)
    partition = f"PARTITION BY RANGE(order_date) ({partitions})" if table == "ext_partitioned" else ""
    ddl = f'CREATE TABLE {table} ({fields}) DUPLICATE KEY(order_date,order_id) {partition} DISTRIBUTED BY HASH(order_id) BUCKETS 4 PROPERTIES("replication_num"="1")'
    show_sql("物理设计", ddl)
    lab.execute(ddl)
source = """SELECT DATE_ADD(CAST('2026-01-01' AS DATE), INTERVAL CAST(FLOOR(number / 10000) AS INT) DAY),
number, CAST(10 AS DECIMAL(12,2)) FROM numbers("number"="100000")"""
previous_group = lab.query("SELECT @@group_commit")[0][0]
try:
    lab.execute("SET group_commit = 'off_mode'")
    for table, batch_size in (("ext_small_batches",1000), ("ext_large_batch",100000), ("ext_partitioned",100000)):
        start = perf_counter()
        for offset in range(0,100000,batch_size):
            lab.execute(f"INSERT INTO {table} {source} WHERE number >= {offset} AND number < {offset+batch_size}")
        print(table, "提交批次",100000//batch_size, "写入秒数",round(perf_counter()-start,3))
        expect(lab.query(f"SELECT COUNT(*), SUM(amount),COUNT(DISTINCT order_id) FROM {table}"), [(100000,"1000000.00",100000)])
        lab.sql(f"SHOW TABLETS FROM {table}", title="写入后 Tablet 状态（不是固定预期值）")
finally:
    lab.execute("SET group_commit = %s", (previous_group,))
for table in ("ext_small_batches", "ext_partitioned"):
    expect(lab.query(f"SELECT COUNT(*) FROM {table} a FULL OUTER JOIN ext_large_batch b ON a.order_id=b.order_id WHERE a.order_id IS NULL OR b.order_id IS NULL OR a.order_date<>b.order_date OR a.amount<>b.amount"), [(0,)])

## 2. 结果、裁剪和 Profile

同一谓词分别在无分区和有分区表运行。先预热一次，再交替运行五轮，保留每轮耗时；这不是并发基准或 P95。
`EXPLAIN` 查看分区/Tablet 裁剪，Profile 查看实际扫描行数及算子耗时。查询输出相同不代表扫描量相同。
可按 Course 2 的 `SHOW TABLET → DetailCmd → CompactionStatus` 入口继续读 Rowset；禁止触发手工 Compaction。

In [ ]:
predicate = "order_date = '2026-01-03'"
previous_profile = lab.query("SELECT @@enable_profile")[0][0]
try:
    lab.execute("SET enable_profile = true")
    for table in ("ext_large_batch", "ext_partitioned"):
        lab.sql(f"EXPLAIN SELECT SUM(amount) FROM {table} WHERE {predicate}", title=table + " 计划")
        expect(lab.query(f"SELECT COUNT(*), SUM(amount) FROM {table} WHERE {predicate}"), [(10000,"100000.00")])
    for iteration in range(5):
        for table in ("ext_large_batch", "ext_partitioned"):
            start = perf_counter()
            expect(lab.query(f"SELECT COUNT(*), SUM(amount) FROM {table} WHERE {predicate}"), [(10000,"100000.00")])
            print(iteration+1,table,round((perf_counter()-start)*1000,3),"ms")
    lab.sql("SHOW QUERY PROFILE", title="按 SQL、库名及时间查找本实验 Profile")
finally:
    lab.execute("SET enable_profile = %s", (previous_profile,))

## 3. 独立解释与排查

记录两张表计划中的 partitions/tablets 字段，从 Profile 找到对应扫描算子；如果耗时没有下降，结合数据规模、缓存和固定查询开销解释，不更改结果。
将小批次改为 5,000 行，重新运行本实验，比较实际批次数和 VersionCount，而不是要求版本数严格等于批次数。
如 Profile 列表暂未出现记录，稍后重新执行 `SHOW QUERY PROFILE`；详细字段阅读见 [Query Profile](https://doris.apache.org/docs/4.x/query-acceleration/query-profile/)。
会话开关由 finally 恢复；结束时保留三张表供排查，不删除数据。

In [ ]:
lab.close()